# Laboratorio 7: NLP y embeddings

Pipeline reproducible para texto, SGNS, analogías y clasificación de noticias.

## Investigación de herramientas y conceptos

### `nn.Embedding`

Esta capa funciona como una tabla de consulta entrenable. `num_embeddings` define la cantidad de entradas del vocabulario. `embedding_dim` define la longitud de cada vector. `padding_idx` reserva una entrada cuyo gradiente no se actualiza. `sparse` permite gradientes dispersos y reduce trabajo cuando solo se consultan pocas filas.

### `nn.Embedding.from_pretrained`

Este método construye una tabla a partir de vectores existentes. El argumento `freeze` decide si los vectores permanecen fijos o reciben ajuste fino durante la tarea supervisada. Esta opción permite comparar representaciones congeladas y entrenables bajo el mismo clasificador.

### `nn.EmbeddingBag`

Esta capa combina consultas y agregación sin crear secuencias rellenadas. `offsets` indica dónde comienza cada documento dentro de un arreglo plano de identificadores. El modo `mean` calcula el promedio y es adecuado para representar noticias con distinta longitud. El modo `sum` conserva la magnitud asociada a la cantidad de tokens. El modo `max` conserva el valor máximo por dimensión.

### CBOW y skip gram

CBOW usa las palabras del contexto para predecir la palabra central. Suele ser rápido y estable para palabras frecuentes. Skip gram usa la palabra central para predecir palabras de contexto. Produce más ejemplos y suele representar mejor palabras menos frecuentes.

### Dos matrices en Word2Vec

Word2Vec mantiene una matriz de entrada `W` para palabras centrales y una matriz de salida `W prima` para palabras de contexto. Cada matriz aprende un papel distinto dentro de la tarea predictiva. Para usos posteriores normalmente se conserva `W`, aunque también es posible combinar ambas matrices si la evaluación lo justifica.

### Word2Vec y GloVe

Word2Vec aprende mediante una tarea predictiva local sobre pares de palabras. SGNS aproxima esta tarea con ejemplos positivos y negativos. GloVe parte de estadísticas globales de coocurrencia y ajusta vectores para representar razones entre esas frecuencias. La comparación debe considerar que GloVe usa un corpus mucho mayor y un procedimiento de entrenamiento diferente.

In [ ]:
from collections import Counter
from functools import partial
import json
import os
from pathlib import Path
import random
import re
import sys
sys.path.insert(0, str(Path.cwd() / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from embeddings_lab.analogies import analogy_3cosadd, analogy_3cosmul, answer_rank, evaluate_analogies, most_similar, tsne_projection
from embeddings_lab.classification import EmbeddingBagClassifier, embedding_bag_collate, encode_document, evaluate_classifier, stratified_split, train_classifier
from embeddings_lab.experiments import ExperimentConfig, count_model_parameters, count_skipgram_pairs, oov_rate, save_run_metadata, seed_everything
from embeddings_lab.preprocessing import build_vocabulary, iter_skipgram_pairs_from_sequences, normalize_text, subsample_ids, subsampling_keep_probabilities, word_tokenize
from embeddings_lab.sgns import SGNS, train_sgns


## Configuración reproducible

In [ ]:
RUN_FULL = os.getenv('EMBEDDINGS_LAB_FULL', '0') == '1'
config = ExperimentConfig()
seed_everything(config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(config.output_dir).mkdir(parents=True, exist_ok=True)
print({'run_full': RUN_FULL, 'device': str(device), **config.to_dict()})


## Exploración del corpus

In [ ]:
if RUN_FULL:
    from datasets import load_dataset
    wiki_rows = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train', streaming=True, cache_dir='data/huggingface')
else:
    wiki_rows = [
        {'text': '= Language =\nThe king and queen rule the country.'},
        {'text': 'A woman and a man write computer programs.'},
        {'text': '= Science =\nGood models learn from repeated context.'},
    ] * 40

token_limit = config.corpus_token_limit if RUN_FULL else 2_000
processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)
corpus_path = processed_dir / 'wikitext_subset.txt'
token_counts = Counter()
articles = 0
lines = 0
tokens = 0
header_pattern = re.compile(r'^=\s+[^=].*?\s+=$')
with corpus_path.open('w', encoding='utf-8') as corpus_file:
    for row in wiki_rows:
        for line in row['text'].splitlines() or [row['text']]:
            lines += 1
            articles += bool(header_pattern.fullmatch(line.strip()))
            sequence = word_tokenize(line)
            if not sequence:
                continue
            remaining = token_limit - tokens
            sequence = sequence[:remaining]
            corpus_file.write(' '.join(sequence) + '\n')
            token_counts.update(sequence)
            tokens += len(sequence)
            if tokens >= token_limit:
                break
        if tokens >= token_limit:
            break
corpus_summary = {'articles': articles, 'lines': lines, 'tokens': tokens, 'unique_tokens': len(token_counts)}
pd.DataFrame([corpus_summary])


## Tokenización

In [ ]:
from nltk.tokenize import TreebankWordTokenizer
library_tokenizer = TreebankWordTokenizer()
examples = [
    "I can't believe it's working.",
    'State-of-the-art systems change quickly.',
    'Dr. Smith arrived at 3.30 p.m.',
    'The price is 1,234.50 dollars.',
    'New York-based teams won.',
    'Email test@example.com today.',
    '<unk> is a reserved token.',
    'Children played, laughed and ran.',
    'Version 2.0-beta is available.',
    'France and Germany are countries.',
]
tokenizer_comparison = pd.DataFrame([
    {'text': text, 'custom': word_tokenize(text), 'library': library_tokenizer.tokenize(normalize_text(text))}
    for text in examples
])
tokenizer_comparison


## Ley de Zipf y vocabulario

In [ ]:
ranked_counts = np.asarray([count for _, count in token_counts.most_common()], dtype=np.int64)
ranks = np.arange(1, len(ranked_counts) + 1)
if len(ranked_counts):
    plt.figure(figsize=(7, 4))
    plt.loglog(ranks, ranked_counts)
    plt.xlabel('Rango')
    plt.ylabel('Frecuencia')
    plt.title('Frecuencia frente a rango')
    plt.show()
coverage_rows = []
for k in [10, 1_000, 30_000]:
    coverage_rows.append({'top_k': k, 'token_coverage': ranked_counts[:k].sum() / ranked_counts.sum() if ranked_counts.size else 0.0})
threshold_rows = []
for minimum in [2, 5, 10]:
    vocabulary_size = 1 + sum(count >= minimum for count in token_counts.values())
    known = sum(count for count in token_counts.values() if count >= minimum)
    threshold_rows.append({'min_count': minimum, 'vocabulary_size': vocabulary_size, 'oov_rate': 1 - known / tokens if tokens else 0.0})
display(pd.DataFrame(coverage_rows))
display(pd.DataFrame(threshold_rows))
active_min_count = config.min_count if RUN_FULL else 1
kept_tokens = sorted((token for token, count in token_counts.items() if count >= active_min_count), key=lambda token: (-token_counts[token], token))
vocabulary = {'<unk>': 0, **{token: index for index, token in enumerate(kept_tokens, 1)}}
id_to_token = [None] * len(vocabulary)
for token, token_id in vocabulary.items():
    id_to_token[token_id] = token
(processed_dir / 'vocabulary.json').write_text(json.dumps(vocabulary, ensure_ascii=False), encoding='utf-8')


## Submuestreo y pares skip gram

In [ ]:
with corpus_path.open(encoding='utf-8') as corpus_file:
    encoded_sequences = [[vocabulary.get(token, 0) for token in line.split()] for line in corpus_file]
keep_by_token = subsampling_keep_probabilities(token_counts, config.subsampling_threshold)
keep_by_id = [1.0] * len(vocabulary)
for token, token_id in vocabulary.items():
    keep_by_id[token_id] = keep_by_token.get(token, 1.0)
rng = random.Random(config.seed)
if RUN_FULL:
    subsampled_sequences = [list(subsample_ids(sequence, keep_by_id, rng=rng)) for sequence in encoded_sequences]
else:
    subsampled_sequences = encoded_sequences
discard_rows = []
for word in ['the', 'of', 'and']:
    token_id = vocabulary.get(word)
    original_count = sum(sequence.count(token_id) for sequence in encoded_sequences) if token_id is not None else 0
    retained_count = sum(sequence.count(token_id) for sequence in subsampled_sequences) if token_id is not None else 0
    observed = 100 * (original_count - retained_count) / original_count if original_count else 0.0
    discard_rows.append({'word': word, 'expected_discard_percent': 100 * (1 - keep_by_token.get(word, 1.0)), 'observed_discard_percent': observed})
pairs_before = sum(count_skipgram_pairs(sequence, config.window_size) for sequence in encoded_sequences)
pairs_after = sum(count_skipgram_pairs(sequence, config.window_size) for sequence in subsampled_sequences)
display(pd.DataFrame(discard_rows))
pair_counts = pd.DataFrame([{'pairs_before': pairs_before, 'pairs_after': pairs_after}])
pair_counts.to_csv(Path(config.output_dir) / 'pair_counts.csv', index=False)
del encoded_sequences
pair_counts


## Entrenamiento SGNS

In [ ]:
def pair_factory():
    return iter_skipgram_pairs_from_sequences(subsampled_sequences, config.window_size)

counts_by_id = np.zeros(len(vocabulary), dtype=np.float64)
for token, token_id in vocabulary.items():
    counts_by_id[token_id] = token_counts.get(token, 0)

wordsim_records = []
analogy_records = []
if RUN_FULL:
    from gensim.test.utils import datapath
    with open(datapath('wordsim353.tsv'), encoding='utf-8') as handle:
        for line in handle:
            if line.startswith('#'):
                continue
            left, right, score = line.rstrip().split('\t')
            wordsim_records.append((left.lower(), right.lower(), float(score)))
    category = None
    with open(datapath('questions-words.txt'), encoding='utf-8') as handle:
        for line in handle:
            if line.startswith(':'):
                category = line[1:].strip()
            else:
                analogy_records.append((category, *line.lower().split()))

semantic_categories = {'capital-common-countries', 'capital-world', 'currency', 'city-in-state', 'family'}
control_words = ['king', 'france', 'computer', 'good', 'january', 'run']
def epoch_evaluation(epoch, model, metrics):
    vectors = model.embeddings.detach().cpu().numpy()
    observed = []
    expected = []
    for left, right, score in wordsim_records:
        if left in vocabulary and right in vocabulary:
            first = vectors[vocabulary[left]]
            second = vectors[vocabulary[right]]
            denominator = np.linalg.norm(first) * np.linalg.norm(second)
            observed.append(float(first @ second / denominator) if denominator else 0.0)
            expected.append(score)
    semantic = [record[1:] for record in analogy_records if record[0] in semantic_categories]
    syntactic = [record[1:] for record in analogy_records if record[0] not in semantic_categories]
    semantic_metrics = evaluate_analogies(vectors, vocabulary, semantic) if semantic else {'accuracy': 0.0, 'evaluated': 0}
    syntactic_metrics = evaluate_analogies(vectors, vocabulary, syntactic) if syntactic else {'accuracy': 0.0, 'evaluated': 0}
    evaluated_total = semantic_metrics['evaluated'] + syntactic_metrics['evaluated']
    total_accuracy = (semantic_metrics['accuracy'] * semantic_metrics['evaluated'] + syntactic_metrics['accuracy'] * syntactic_metrics['evaluated']) / evaluated_total if evaluated_total else 0.0
    neighbors = {word: most_similar(vectors, vocabulary, word, top_k=5) for word in control_words if word in vocabulary}
    return {
        'wordsim_spearman': float(spearmanr(observed, expected).statistic) if len(observed) > 1 else 0.0,
        'semantic_accuracy': semantic_metrics['accuracy'],
        'syntactic_accuracy': syntactic_metrics['accuracy'],
        'total_accuracy': total_accuracy,
        'neighbors': neighbors,
    }

dimensions = config.dimensions if RUN_FULL else (8,)
epochs = config.epochs if RUN_FULL else 2
embedding_runs = {}
history_rows = []
for dimension in dimensions:
    model = SGNS(len(vocabulary), dimension, seed=config.seed)
    result = train_sgns(
        model, pair_factory, counts_by_id, epochs=epochs, batch_size=config.batch_size if RUN_FULL else 64,
        num_negatives=config.num_negatives, learning_rate=config.learning_rate, seed=config.seed,
        device=device, reserved_ids=(0,), epoch_callback=epoch_evaluation,
    )
    vectors = model.embeddings.detach().cpu().numpy()
    embedding_runs[dimension] = {'model': model, 'vectors': vectors, 'result': result}
    for row in result['epoch_records']:
        history_rows.append({'dimension': dimension, **row})
    run_dir = Path(config.output_dir) / f'sgns_dim_{dimension}'
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), run_dir / 'model.pt')
    np.save(run_dir / 'vectors.npy', vectors)
    save_run_metadata(config.output_dir, f'sgns_dim_{dimension}', {'dimension': dimension, 'tokens': tokens, 'epochs': epochs, 'min_count': active_min_count, 'total_time_seconds': result['total_time_seconds'], 'peak_gpu_memory_bytes': result['peak_gpu_memory_bytes']})
history = pd.DataFrame(history_rows)
history.to_csv(Path(config.output_dir) / 'sgns_history.csv', index=False)
run_summary = pd.DataFrame([{'dimension': dimension, 'total_time_seconds': run['result']['total_time_seconds'], 'peak_gpu_memory_bytes': run['result']['peak_gpu_memory_bytes'], **{key: run['result']['epoch_records'][-1].get(key, 0.0) for key in ['wordsim_spearman', 'semantic_accuracy', 'syntactic_accuracy', 'total_accuracy']}} for dimension, run in embedding_runs.items()])
run_summary['selection_score'] = run_summary['wordsim_spearman'].fillna(-1) + run_summary['total_accuracy'].fillna(0)
best_dimension = int(run_summary.sort_values(['selection_score', 'dimension'], ascending=[False, True]).iloc[0]['dimension'])
reference_dimension = 100 if 100 in embedding_runs else best_dimension
run_summary.to_csv(Path(config.output_dir) / 'sgns_summary.csv', index=False)
history


In [ ]:
if not history.empty:
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    for dimension, group in history.groupby('dimension'):
        axes[0].plot(group['epoch'], group['loss'], marker='o', label=str(dimension))
        axes[1].plot(group['epoch'], group['total_accuracy'], marker='o', label=str(dimension))
    axes[0].set_title('Pérdida por época')
    axes[1].set_title('Accuracy de analogías por época')
    for axis in axes:
        axis.set_xlabel('Época')
        axis.legend(title='Dimensión')
    plt.show()


## Analogías y visualización

In [ ]:
custom_analogies = [
    ('man', 'king', 'woman', 'queen', 'género'), ('boy', 'prince', 'girl', 'princess', 'género'), ('father', 'man', 'mother', 'woman', 'género'),
    ('france', 'paris', 'germany', 'berlin', 'capital'), ('italy', 'rome', 'japan', 'tokyo', 'capital'), ('spain', 'madrid', 'portugal', 'lisbon', 'capital'),
    ('france', 'french', 'germany', 'german', 'gentilicio'), ('italy', 'italian', 'spain', 'spanish', 'gentilicio'), ('japan', 'japanese', 'china', 'chinese', 'gentilicio'),
    ('good', 'better', 'bad', 'worse', 'comparativo'), ('good', 'best', 'bad', 'worst', 'superlativo'), ('small', 'smaller', 'large', 'larger', 'comparativo'),
    ('walk', 'walked', 'run', 'ran', 'verbo'), ('car', 'cars', 'book', 'books', 'plural'), ('child', 'children', 'man', 'men', 'plural'),
]
final_dimension = reference_dimension
final_vectors = embedding_runs[final_dimension]['vectors']
def analogia(a, b, c, k):
    return analogy_3cosadd(final_vectors, vocabulary, a, b, c, top_k=k)

from gensim.models import KeyedVectors
keyed_vectors = KeyedVectors(vector_size=final_dimension)
keyed_vectors.add_vectors(id_to_token, final_vectors)
vector_path = Path(config.output_dir) / 'final_sgns_vectors.txt'
keyed_vectors.save_word2vec_format(str(vector_path), binary=False)
analogy_rows = []
for a, b, c, expected, category in custom_analogies:
    missing = [word for word in [a, b, c, expected] if word not in vocabulary]
    if missing:
        analogy_rows.append({'category': category, 'query': f'{a}:{b}::{c}', 'expected': expected, 'status': 'oov', 'missing': missing})
        continue
    ranked = analogia(a, b, c, len(vocabulary))
    multiplied = analogy_3cosmul(final_vectors, vocabulary, a, b, c, top_k=5)
    unfiltered = analogy_3cosadd(final_vectors, vocabulary, a, b, c, top_k=5, exclude_inputs=False)
    va, vb, vc, vd = [final_vectors[vocabulary[word]] for word in [a, b, c, expected]]
    query = vb / np.linalg.norm(vb) - va / np.linalg.norm(va) + vc / np.linalg.norm(vc)
    answer_similarity = float(query @ vd / (np.linalg.norm(query) * np.linalg.norm(vd)))
    gensim_top = keyed_vectors.most_similar(positive=[b, c], negative=[a], topn=5)
    analogy_rows.append({'category': category, 'query': f'{a}:{b}::{c}', 'expected': expected, 'status': 'evaluated', 'top_5_3cosadd': ranked[:5], 'top_5_3cosmul': multiplied, 'gensim_top_5': gensim_top, 'same_top_1_as_gensim': ranked[0][0] == gensim_top[0][0], 'answer_rank': answer_rank(ranked, expected), 'answer_similarity': answer_similarity, 'without_exclusion': unfiltered})
analogy_table = pd.DataFrame(analogy_rows)
analogy_table.to_json(Path(config.output_dir) / 'individual_analogies.json', orient='records', force_ascii=False, indent=2)
analogy_table


In [ ]:
eligible_words = [word for word in id_to_token if word != '<unk>'][:500]
if len(eligible_words) >= 2:
    selected_vectors = np.asarray([final_vectors[vocabulary[word]] for word in eligible_words])
    coordinates = tsne_projection(selected_vectors, seed=config.seed)
    plt.figure(figsize=(12, 9))
    plt.scatter(coordinates[:, 0], coordinates[:, 1], s=8, alpha=0.6)
    for index in range(0, len(eligible_words), max(1, len(eligible_words) // 40)):
        plt.annotate(eligible_words[index], coordinates[index], fontsize=7)
    plt.title('Proyección t-SNE de palabras frecuentes')
    plt.show()


## Clasificación de AG News

In [ ]:
if RUN_FULL:
    news = load_dataset('fancyzhx/ag_news', cache_dir='data/huggingface')
    all_train_texts = list(news['train']['text'])
    all_train_labels = list(news['train']['label'])
    test_texts = list(news['test']['text'])
    test_labels = list(news['test']['label'])
else:
    templates = ['world leaders meet in city', 'team wins sports match', 'market reports company profit', 'new computer technology released']
    all_train_texts = [templates[label] + f' news {index}' for label in range(4) for index in range(20)]
    all_train_labels = [label for label in range(4) for _ in range(20)]
    test_texts = [templates[label] for label in range(4) for _ in range(4)]
    test_labels = [label for label in range(4) for _ in range(4)]
train_texts, validation_texts, train_labels, validation_labels = stratified_split(all_train_texts, all_train_labels, seed=config.seed)
split_path = processed_dir / 'ag_news_splits.jsonl'
with split_path.open('w', encoding='utf-8') as handle:
    for split_name, texts_part, labels_part in [('train', train_texts, train_labels), ('validation', validation_texts, validation_labels), ('test', test_texts, test_labels)]:
        for text, label in zip(texts_part, labels_part):
            handle.write(json.dumps({'split': split_name, 'text': text, 'label': label}, ensure_ascii=False) + '\n')
news_oov = oov_rate((word_tokenize(text) for text in all_train_texts), vocabulary)
def encoded_dataset(texts, labels):
    return [(encode_document(text, vocabulary), label) for text, label in zip(texts, labels)]
collate = partial(embedding_bag_collate, unknown_id=vocabulary['<unk>'])
train_loader = DataLoader(encoded_dataset(train_texts, train_labels), batch_size=128 if RUN_FULL else 16, shuffle=True, collate_fn=collate)
validation_loader = DataLoader(encoded_dataset(validation_texts, validation_labels), batch_size=256, shuffle=False, collate_fn=collate)
test_loader = DataLoader(encoded_dataset(test_texts, test_labels), batch_size=256, shuffle=False, collate_fn=collate)
news_oov


In [ ]:
classifier_epochs = 5 if RUN_FULL else 2
pretrained = torch.tensor(final_vectors, dtype=torch.float32)
variant_builders = {
    'random': lambda: EmbeddingBagClassifier(len(vocabulary), final_dimension, hidden_dim=128, dropout=0.2),
    'sgns_frozen': lambda: EmbeddingBagClassifier(len(vocabulary), final_dimension, pretrained_embeddings=pretrained, freeze=True, hidden_dim=128, dropout=0.2),
    'sgns_finetuned': lambda: EmbeddingBagClassifier(len(vocabulary), final_dimension, pretrained_embeddings=pretrained, freeze=False, hidden_dim=128, dropout=0.2),
}
classifier_rows = []
classifier_history_rows = []
trained_classifiers = {}
for name, builder in variant_builders.items():
    seed_everything(config.seed)
    classifier = builder()
    classifier_history = train_classifier(classifier, train_loader, validation_loader, epochs=classifier_epochs, device=device)
    validation_metrics = evaluate_classifier(classifier, validation_loader, device=device)
    classifier_rows.append({'variant': name, **count_model_parameters(classifier), **validation_metrics, 'training_seconds': sum(row['duration_seconds'] for row in classifier_history)})
    classifier_history_rows.extend({'variant': name, **row} for row in classifier_history)
    trained_classifiers[name] = {'model': classifier, 'history': classifier_history}
classification_results = pd.DataFrame(classifier_rows).sort_values('macro_f1', ascending=False)
classification_history = pd.DataFrame(classifier_history_rows)
classification_results.to_csv(Path(config.output_dir) / 'classification_validation.csv', index=False)
classification_history.to_csv(Path(config.output_dir) / 'classification_history.csv', index=False)
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, group in classification_history.groupby('variant'):
    axes[0].plot(group['epoch'], group['train_loss'], marker='o', label=name)
    axes[1].plot(group['epoch'], group['validation_loss'], marker='o', label=name)
axes[0].set_title('Pérdida de entrenamiento')
axes[1].set_title('Pérdida de validación')
for axis in axes:
    axis.set_xlabel('Época')
    axis.legend()
plt.show()
fraction_rows = []
fractions = [0.01, 0.25, 0.5, 1.0] if RUN_FULL else [1.0]
sgns_choice = classification_results[classification_results['variant'].str.startswith('sgns')].iloc[0]['variant']
for fraction in fractions:
    if fraction < 1.0:
        fraction_texts, _, fraction_labels, _ = train_test_split(train_texts, train_labels, train_size=fraction, stratify=train_labels, random_state=config.seed)
    else:
        fraction_texts, fraction_labels = train_texts, train_labels
    fraction_loader = DataLoader(encoded_dataset(fraction_texts, fraction_labels), batch_size=128 if RUN_FULL else 16, shuffle=True, collate_fn=collate)
    for name in ['random', sgns_choice]:
        seed_everything(config.seed)
        fraction_model = variant_builders[name]()
        fraction_history = train_classifier(fraction_model, fraction_loader, validation_loader, epochs=classifier_epochs, device=device)
        fraction_rows.append({'fraction': fraction, 'initialization': 'random' if name == 'random' else 'sgns', **count_model_parameters(fraction_model), 'training_seconds': sum(row['duration_seconds'] for row in fraction_history), 'history': fraction_history, **evaluate_classifier(fraction_model, validation_loader, device=device)})
fraction_results = pd.DataFrame(fraction_rows)
fraction_results.drop(columns=['history']).to_csv(Path(config.output_dir) / 'classification_fractions.csv', index=False)
(Path(config.output_dir) / 'classification_fraction_histories.json').write_text(json.dumps(fraction_rows, indent=2), encoding='utf-8')
for initialization, group in fraction_results.groupby('initialization'):
    plt.plot(group['fraction'], group['macro_f1'], marker='o', label=initialization)
plt.xlabel('Fracción de entrenamiento')
plt.ylabel('F1 macro de validación')
plt.legend()
plt.show()
classification_results


In [ ]:
def evaluate_test_once(model):
    targets = []
    predictions = []
    total_loss = 0.0
    model.train(False)
    with torch.no_grad():
        for token_ids, offsets, labels in test_loader:
            logits = model(token_ids.to(device), offsets.to(device))
            total_loss += float(torch.nn.functional.cross_entropy(logits, labels.to(device), reduction='sum'))
            targets.extend(labels.tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    precision, recall, f1, _ = precision_recall_fscore_support(targets, predictions, labels=[0, 1, 2, 3], average='macro', zero_division=0)
    metrics = {'loss': total_loss / len(targets), 'accuracy': accuracy_score(targets, predictions), 'macro_precision': precision, 'macro_recall': recall, 'macro_f1': f1, 'count': len(targets)}
    return metrics, confusion_matrix(targets, predictions, labels=[0, 1, 2, 3])

selected_variants = {'random': 'random', 'sgns': sgns_choice}
test_rows = []
test_matrices = {}
figure, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, (initialization, variant) in zip(axes, selected_variants.items()):
    metrics, matrix = evaluate_test_once(trained_classifiers[variant]['model'])
    test_rows.append({'initialization': initialization, 'variant': variant, **metrics})
    test_matrices[initialization] = matrix.tolist()
    ConfusionMatrixDisplay(matrix).plot(cmap='Blues', ax=axis, colorbar=False)
    axis.set_title(initialization)
plt.show()
test_results = pd.DataFrame(test_rows)
test_results.to_csv(Path(config.output_dir) / 'classification_test.csv', index=False)
(Path(config.output_dir) / 'confusion_matrices.json').write_text(json.dumps(test_matrices, indent=2), encoding='utf-8')
test_results


## Modelos de referencia y evaluación sistemática

Esta parte entrena Word2Vec de gensim sobre el mismo corpus y el mismo vocabulario que SGNS, carga GloVe sin modificarlo y evalúa los tres conjuntos de vectores con el mismo protocolo. Las tablas y las gráficas salen de los archivos de métricas guardados en `artifacts/`. Si un archivo ya existe se reutiliza, y para repetir una corrida basta con borrarlo.

In [ ]:
# Herramientas, datos de evaluación y vectores preentrenados
import platform
from gensim.models import KeyedVectors
from gensim.test.utils import datapath
from sklearn.metrics import log_loss

from embeddings_lab.benchmark import (
    CONTROL_WORDS, difference_parallelism, epoch_evaluation, evaluate_by_category, load_analogy_records,
    load_similarity_pairs, restrict_space, select_thematic_words, shared_vocabulary, spearman_similarity,
)
from embeddings_lab.reference_classification import (
    TfidfBaseline, classification_metrics, evaluate_once, pretrained_classifier, stratified_fraction,
)
from embeddings_lab.reference_models import (
    glove_embedding_table, load_encoded_sentences, load_glove, token_total, train_gensim_skipgram,
)

output_dir = Path(config.output_dir)
gensim_dir = output_dir / 'gensim'
figures_dir = output_dir / 'figures'
for directory in (gensim_dir, figures_dir):
    directory.mkdir(parents=True, exist_ok=True)

# WordSim-353 y questions-words.txt sirven para elegir configuraciones.
# SimLex-999 se carga hasta la evaluación final.
wordsim_pairs = load_similarity_pairs(datapath('wordsim353.tsv'))
analogy_records = load_analogy_records(datapath('questions-words.txt'))
hardware_cpu = f'{platform.processor() or platform.machine()}, {os.cpu_count()} hilos lógicos (entrenamiento con 1 hilo)'

# GloVe se usa tal cual, sin entrenamiento adicional.
glove = load_glove('glove-wiki-gigaword-100')
glove_vocabulary = glove.key_to_index
glove_vectors = glove.vectors
print({'wordsim_pairs': len(wordsim_pairs), 'analogies': len(analogy_records),
       'glove_words': len(glove_vocabulary), 'glove_dimension': glove.vector_size})

### Word2Vec de gensim

El modelo de gensim recibe las mismas oraciones ya codificadas que usa SGNS, donde toda palabra fuera del vocabulario se reemplaza por `<unk>`. Se usan los mismos valores de dimensión, ventana, negativos, `min_count`, umbral de submuestreo, épocas y semilla. La ventana es fija (`shrink_windows=False`) igual que en SGNS y el entrenamiento usa un solo hilo para que la corrida se pueda repetir. El optimizador no se puede igualar, porque SGNS usa Adam disperso y gensim usa descenso por gradiente con tasa decreciente.

Además de la configuración principal se entrenan tres tamaños de corpus (25 %, 50 % y 100 % de los tokens) y variaciones de ventana y de negativos sobre el corpus completo. Al final de cada época se calculan las analogías, WordSim-353 y los cinco vecinos de las palabras de control.

In [ ]:
sentences = load_encoded_sentences(corpus_path, vocabulary)
corpus_tokens = token_total(sentences)
del sentences  # cada corrida vuelve a leer su porción del corpus, así no se mantiene todo en memoria
gensim_epochs = config.epochs if RUN_FULL else 2
fraction_runs = [0.25, 0.5, 1.0] if RUN_FULL else [1.0]
run_specs = {f'corpus_{int(fraction * 100)}': {'fraction': fraction} for fraction in fraction_runs}
if RUN_FULL:
    run_specs.update({
        'window_2': {'window': 2}, 'window_10': {'window': 10},
        'negatives_2': {'negatives': 2}, 'negatives_15': {'negatives': 15},
    })

def gensim_epoch_metrics(epoch, keyed_vectors):
    return epoch_evaluation(keyed_vectors.vectors, keyed_vectors.key_to_index, wordsim_pairs, analogy_records)

for run_name, spec in run_specs.items():
    run_dir = gensim_dir / run_name
    summary_path = run_dir / 'summary.json'
    if summary_path.exists():
        continue
    run_dir.mkdir(parents=True, exist_ok=True)
    window = spec.get('window', config.window_size)
    negatives = spec.get('negatives', config.num_negatives)
    subset = load_encoded_sentences(corpus_path, vocabulary, max_tokens=int(corpus_tokens * spec.get('fraction', 1.0)))
    result = train_gensim_skipgram(
        subset, dimension=final_dimension, window=window, negatives=negatives, min_count=active_min_count,
        sample=config.subsampling_threshold, epochs=gensim_epochs, seed=config.seed, workers=1,
        epoch_evaluator=gensim_epoch_metrics,
    )
    model = result['model']
    model.wv.save(str(run_dir / 'vectors.kv'))
    last = result['epoch_records'][-1]
    (run_dir / 'history.json').write_text(json.dumps(result['epoch_records'], ensure_ascii=False, indent=2), encoding='utf-8')
    summary_path.write_text(json.dumps({
        'run': run_name, 'fraction': spec.get('fraction', 1.0), 'tokens': result['tokens'],
        'vocabulary_size': len(model.wv), 'dimension': final_dimension, 'window': window, 'negatives': negatives,
        'min_count': active_min_count, 'sample': config.subsampling_threshold, 'epochs': gensim_epochs,
        'seed': config.seed, 'workers': 1, 'total_time_seconds': result['total_time_seconds'],
        'vocabulary_seconds': result['vocabulary_seconds'], 'peak_memory_bytes': result['peak_memory_bytes'],
        'hardware': hardware_cpu,
        **{key: last[key] for key in ['wordsim_spearman', 'semantic_accuracy', 'syntactic_accuracy', 'total_accuracy']},
    }, indent=2), encoding='utf-8')
    del subset, result, model

gensim_summary = pd.DataFrame([json.loads((gensim_dir / name / 'summary.json').read_text(encoding='utf-8')) for name in run_specs])
gensim_history = pd.DataFrame([
    {'run': name, **{key: value for key, value in record.items() if key != 'neighbors'}}
    for name in run_specs
    for record in json.loads((gensim_dir / name / 'history.json').read_text(encoding='utf-8'))
])
gensim_summary.to_csv(output_dir / 'gensim_summary.csv', index=False)
gensim_history.to_csv(output_dir / 'gensim_history.csv', index=False)
gensim_final_text = output_dir / 'gensim_final_vectors.txt'
if not gensim_final_text.exists():
    KeyedVectors.load(str(gensim_dir / 'corpus_100' / 'vectors.kv')).save_word2vec_format(str(gensim_final_text), binary=False)
gensim_summary

In [ ]:
# Curvas por época de las iteraciones de gensim y vecinos de las palabras de control
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for run_name, group in gensim_history.groupby('run'):
    axes[0].plot(group['epoch'], group['total_accuracy'], marker='o', label=run_name)
    axes[1].plot(group['epoch'], group['wordsim_spearman'], marker='o', label=run_name)
axes[0].set_title('Accuracy total de analogías por época')
axes[1].set_title('Spearman en WordSim-353 por época')
for axis in axes:
    axis.set_xlabel('Época')
    axis.legend(title='Iteración', fontsize=7)
plt.savefig(figures_dir / 'gensim_epoch_curves.png', dpi=150, bbox_inches='tight')
plt.show()

final_history = json.loads((gensim_dir / 'corpus_100' / 'history.json').read_text(encoding='utf-8'))
neighbor_table = pd.DataFrame([
    {'epoch': record['epoch'], 'word': word, 'neighbors': ', '.join(name for name, _ in found)}
    for record in final_history for word, found in record['neighbors'].items()
])
neighbor_table.to_csv(output_dir / 'gensim_neighbors.csv', index=False)
neighbor_table[neighbor_table['epoch'] == gensim_epochs]

### Vocabulario compartido y benchmark por categoría

Los tres conjuntos se comparan sobre las 30,000 palabras más frecuentes del corpus que existen en SGNS, en gensim y en GloVe. Así las preguntas evaluadas son las mismas para los tres modelos y la cobertura es igual. El orden de frecuencia es el del vocabulario de SGNS.

In [ ]:
gensim_final = KeyedVectors.load(str(gensim_dir / 'corpus_100' / 'vectors.kv'))
gensim_vocabulary = gensim_final.key_to_index
gensim_vectors = gensim_final.vectors
model_spaces = {
    'sgns': (final_vectors, vocabulary),
    'gensim': (gensim_vectors, gensim_vocabulary),
    'glove': (glove_vectors, glove_vocabulary),
}
shared_words = shared_vocabulary(id_to_token, vocabulary, gensim_vocabulary, glove_vocabulary, size=30_000)
shared_spaces = {name: restrict_space(vectors, vocab, shared_words) for name, (vectors, vocab) in model_spaces.items()}
pd.DataFrame([{
    'sgns_vocabulary': len(vocabulary), 'gensim_vocabulary': len(gensim_vocabulary), 'glove_vocabulary': len(glove_vocabulary),
    'shared_vocabulary': len(shared_words),
    'same_words_sgns_gensim': set(vocabulary) == set(gensim_vocabulary),
    'same_order_sgns_gensim': float(np.mean([left == right for left, right in zip(id_to_token, gensim_final.index_to_key)])),
}])

In [ ]:
benchmark_path = output_dir / 'analogy_benchmark.csv'
if benchmark_path.exists():
    analogy_benchmark = pd.read_csv(benchmark_path)
else:
    frames = []
    for model_name, (vectors, vocab) in shared_spaces.items():
        table = evaluate_by_category(vectors, vocab, analogy_records)
        table.insert(0, 'model', model_name)
        frames.append(table)
    analogy_benchmark = pd.concat(frames, ignore_index=True)
    analogy_benchmark.to_csv(benchmark_path, index=False)

# Con el vocabulario compartido todos los modelos evalúan las mismas preguntas
assert analogy_benchmark.groupby(['method', 'category'])['evaluated'].nunique().max() == 1
category_order = list(dict.fromkeys(record[0] for record in analogy_records)) + ['semantic', 'syntactic', 'total']
coverage_table = analogy_benchmark[(analogy_benchmark['model'] == 'sgns') & (analogy_benchmark['method'] == '3cosadd')]
display(coverage_table.set_index('category').loc[category_order, ['total', 'evaluated', 'coverage']])
accuracy_table = analogy_benchmark.pivot_table(index='category', columns=['method', 'model'], values='accuracy').loc[category_order]
display(accuracy_table)

figure, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for axis, method in zip(axes, ['3cosadd', '3cosmul']):
    accuracy_table[method][['sgns', 'gensim', 'glove']].iloc[:14].plot.bar(ax=axis)
    axis.set_title(f'Accuracy por categoría con {method}')
    axis.set_xlabel('')
axes[0].set_ylabel('Accuracy')
plt.savefig(figures_dir / 'analogy_categories.png', dpi=150, bbox_inches='tight')
plt.show()

### Analogías propias en los tres modelos

Se usan las mismas 15 analogías propias de la sección anterior, buscando la respuesta entre las palabras del vocabulario compartido. Para cada modelo se guarda el top 5 con 3CosAdd y 3CosMul, el rango de la respuesta correcta, la similitud coseno de la respuesta con el vector resultante y el resultado al no excluir las palabras de la consulta. También se compara la lista con `most_similar` de gensim.

In [ ]:
analogies_path = output_dir / 'analogies_all_models.json'
if analogies_path.exists():
    analogy_rows_all = json.loads(analogies_path.read_text(encoding='utf-8'))
else:
    # Las tres comparaciones usan el vocabulario compartido y un KeyedVectors de gensim construido con los mismos vectores
    keyed_models = {}
    for model_name, (vectors, vocab) in shared_spaces.items():
        keyed_models[model_name] = KeyedVectors(vector_size=vectors.shape[1])
        keyed_models[model_name].add_vectors(shared_words, vectors.astype(np.float32))
    analogy_rows_all = []
    for model_name, (vectors, vocab) in shared_spaces.items():
        for a, b, c, expected, category in custom_analogies:
            row = {'model': model_name, 'category': category, 'query': f'{a}:{b}::{c}', 'expected': expected}
            missing = [word for word in (a, b, c, expected) if word not in vocab]
            if missing:
                analogy_rows_all.append({**row, 'status': 'oov', 'missing': missing})
                continue
            size = len(vocab)
            excluded = analogy_3cosadd(vectors, vocab, a, b, c, top_k=size)
            included = analogy_3cosadd(vectors, vocab, a, b, c, top_k=size, exclude_inputs=False)
            product = analogy_3cosmul(vectors, vocab, a, b, c, top_k=5)
            reference = keyed_models[model_name].most_similar(positive=[b, c], negative=[a], topn=5)
            va, vb, vc, vd = (vectors[vocab[word]] for word in (a, b, c, expected))
            query = vb / np.linalg.norm(vb) - va / np.linalg.norm(va) + vc / np.linalg.norm(vc)
            analogy_rows_all.append({
                **row, 'status': 'evaluated',
                'top_5_3cosadd': [[word, score] for word, score in excluded[:5]],
                'top_5_3cosmul': [[word, score] for word, score in product],
                'answer_rank': answer_rank(excluded, expected),
                'answer_similarity': float(query @ vd / (np.linalg.norm(query) * np.linalg.norm(vd))),
                'top_5_without_exclusion': [[word, score] for word, score in included[:5]],
                'answer_rank_without_exclusion': answer_rank(included, expected),
                'gensim_top_5': [[word, float(score)] for word, score in reference],
                'same_top_5_as_gensim': [word for word, _ in excluded[:5]] == [word for word, _ in reference],
            })
    analogies_path.write_text(json.dumps(analogy_rows_all, ensure_ascii=False, indent=2), encoding='utf-8')

analogy_overview = pd.DataFrame([
    {
        'model': row['model'], 'category': row['category'], 'query': row['query'], 'expected': row['expected'],
        'top_1': row['top_5_3cosadd'][0][0], 'rank': row['answer_rank'], 'similarity': row['answer_similarity'],
        'top_1_without_exclusion': row['top_5_without_exclusion'][0][0],
        'rank_without_exclusion': row['answer_rank_without_exclusion'],
        'same_top_5_as_gensim': row['same_top_5_as_gensim'],
    }
    for row in analogy_rows_all if row['status'] == 'evaluated'
])
analogy_overview.to_csv(output_dir / 'analogies_all_models.csv', index=False)
analogy_overview

### Paralelismo de los vectores diferencia

Para cada categoría de `questions-words.txt` se toman los pares (a, b) y (c, d) presentes en el vocabulario compartido y se calcula el coseno promedio entre todos sus vectores diferencia. Como punto de comparación se usa el mismo cálculo con pares de palabras al azar. Un valor alto indica que la relación se codifica como un desplazamiento casi constante.

In [ ]:
parallelism_path = output_dir / 'parallelism.csv'
if parallelism_path.exists():
    parallelism = pd.read_csv(parallelism_path)
else:
    frames = []
    for model_name, (vectors, vocab) in shared_spaces.items():
        table = difference_parallelism(vectors, vocab, analogy_records, seed=config.seed)
        table.insert(0, 'model', model_name)
        frames.append(table)
    parallelism = pd.concat(frames, ignore_index=True)
    parallelism.to_csv(parallelism_path, index=False)

parallelism_by_category = parallelism.pivot_table(index='category', columns='model', values='mean_cosine').loc[category_order[:14]]
display(parallelism_by_category)
parallelism_summary = parallelism.groupby('model')[['mean_cosine', 'random_mean_cosine']].mean()
display(parallelism_summary)
parallelism_by_category[['sgns', 'gensim', 'glove']].plot.bar(figsize=(12, 4))
plt.ylabel('Coseno promedio entre vectores diferencia')
plt.title('Paralelismo por categoría')
plt.savefig(figures_dir / 'parallelism.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualización t-SNE por grupos temáticos

Se proyectan unas 500 palabras repartidas en diez grupos temáticos (países, ciudades, animales, números, familia, profesiones, verbos, cuerpo, comida y tiempo). Solo se usan palabras del vocabulario compartido, así que los tres modelos proyectan exactamente las mismas palabras.

In [ ]:
tsne_path = output_dir / 'tsne_coordinates.csv'
tsne_words, tsne_groups = select_thematic_words(shared_words, per_group=50)
if tsne_path.exists():
    tsne_table = pd.read_csv(tsne_path)
else:
    frames = []
    for model_name, (vectors, vocab) in shared_spaces.items():
        selected = np.asarray([vectors[vocab[word]] for word in tsne_words])
        coordinates = tsne_projection(selected, seed=config.seed)
        frames.append(pd.DataFrame({'model': model_name, 'word': tsne_words, 'group': tsne_groups,
                                    'x': coordinates[:, 0], 'y': coordinates[:, 1]}))
    tsne_table = pd.concat(frames, ignore_index=True)
    tsne_table.to_csv(tsne_path, index=False)

palette = plt.get_cmap('tab10')
group_names = list(dict.fromkeys(tsne_groups))
figure, axes = plt.subplots(1, 3, figsize=(24, 8))
for axis, model_name in zip(axes, ['sgns', 'gensim', 'glove']):
    points = tsne_table[tsne_table['model'] == model_name]
    for color_index, group_name in enumerate(group_names):
        group_points = points[points['group'] == group_name]
        axis.scatter(group_points['x'], group_points['y'], s=14, color=palette(color_index), label=group_name)
        for _, row in group_points.head(5).iterrows():
            axis.annotate(row['word'], (row['x'], row['y']), fontsize=8)
    axis.set_title(f't-SNE de {len(tsne_words)} palabras, modelo {model_name}')
    axis.set_xticks([])
    axis.set_yticks([])
axes[0].legend(fontsize=8, markerscale=1.5)
plt.savefig(figures_dir / 'tsne_groups.png', dpi=150, bbox_inches='tight')
plt.show()

## Clasificación de AG News con TF-IDF y GloVe

El baseline usa TF-IDF sobre palabras y bigramas con regresión logística. Los clasificadores con GloVe son iguales a los de SGNS, con `nn.EmbeddingBag` en modo promedio y un MLP, y se entrenan con la tabla congelada y con fine-tuning. Todos usan el tokenizador de la sección 2 y las mismas particiones. Para GloVe la tabla es el vocabulario completo de 400,000 palabras más una fila `<unk>` que es el promedio de sus vectores. Esa fila se agrega aparte y no modifica ninguno de los vectores originales.

In [ ]:
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']
split_texts = {'train': train_texts, 'validation': validation_texts, 'test': test_texts}
split_labels = {'train': train_labels, 'validation': validation_labels, 'test': test_labels}
tokenized = {split: TfidfBaseline.tokenize(texts) for split, texts in split_texts.items()}

glove_classifier_vocabulary, glove_table = glove_embedding_table(glove)
oov_table = pd.DataFrame([
    {'split': split, 'embedding': name, 'oov_rate': oov_rate(documents, vocab)}
    for split, documents in tokenized.items()
    for name, vocab in {'sgns_and_gensim': vocabulary, 'glove': glove_classifier_vocabulary}.items()
])
oov_table.to_csv(output_dir / 'ag_news_oov.csv', index=False)
display(oov_table.pivot_table(index='split', columns='embedding', values='oov_rate'))

glove_encoded = {
    split: [([glove_classifier_vocabulary.get(token, 0) for token in document] or [0], label)
            for document, label in zip(documents, split_labels[split])]
    for split, documents in tokenized.items() if split != 'test'
}
glove_collate = partial(embedding_bag_collate, unknown_id=0)
train_batch_size = 128 if RUN_FULL else 16

def make_loader(data, *, shuffle):
    return DataLoader(data, batch_size=train_batch_size if shuffle else 256, shuffle=shuffle, collate_fn=glove_collate,
                      generator=torch.Generator().manual_seed(config.seed))

glove_validation_loader = make_loader(glove_encoded['validation'], shuffle=False)

In [ ]:
reference_rows = []
reference_history_rows = []
glove_models = {}
for variant, freeze in {'glove_frozen': True, 'glove_finetuned': False}.items():
    seed_everything(config.seed)
    model = pretrained_classifier(glove_classifier_vocabulary, glove_table, freeze=freeze)
    history_rows = train_classifier(model, make_loader(glove_encoded['train'], shuffle=True), glove_validation_loader,
                                    epochs=classifier_epochs, device=device)
    validation = evaluate_classifier(model, glove_validation_loader, device=device)
    reference_rows.append({'variant': variant, 'initialization': 'glove', **count_model_parameters(model), **validation,
                           'training_seconds': sum(row['duration_seconds'] for row in history_rows)})
    reference_history_rows.extend({'variant': variant, **row} for row in history_rows)
    glove_models[variant] = model

# Baseline TF-IDF. La regularización se elige con la validación.
tfidf_candidates = {}
for regularization in (1.0, 10.0):
    candidate = TfidfBaseline(regularization=regularization, seed=config.seed).fit(tokenized['train'], split_labels['train'])
    probabilities = candidate.predict_proba(tokenized['validation'])
    metrics = classification_metrics(split_labels['validation'], probabilities.argmax(axis=1))
    metrics['loss'] = float(log_loss(split_labels['validation'], probabilities, labels=[0, 1, 2, 3]))
    tfidf_candidates[regularization] = {'model': candidate, 'metrics': metrics}
best_regularization = max(tfidf_candidates, key=lambda value: tfidf_candidates[value]['metrics']['macro_f1'])
tfidf_model = tfidf_candidates[best_regularization]['model']
reference_rows.append({'variant': 'tfidf_logreg', 'initialization': 'tfidf', **tfidf_model.parameter_counts(),
                       **tfidf_candidates[best_regularization]['metrics'], 'training_seconds': tfidf_model.training_seconds})

reference_validation = pd.DataFrame(reference_rows).sort_values('macro_f1', ascending=False)
reference_history = pd.DataFrame(reference_history_rows)
reference_validation.to_csv(output_dir / 'classification_validation_reference.csv', index=False)
reference_history.to_csv(output_dir / 'classification_history_reference.csv', index=False)
pd.DataFrame([{'regularization': value, **run['metrics']} for value, run in tfidf_candidates.items()]).to_csv(
    output_dir / 'tfidf_regularization.csv', index=False)

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, group in reference_history.groupby('variant'):
    axes[0].plot(group['epoch'], group['train_loss'], marker='o', label=name)
    axes[1].plot(group['epoch'], group['validation_loss'], marker='o', label=name)
axes[0].set_title('Pérdida de entrenamiento')
axes[1].set_title('Pérdida de validación')
for axis in axes:
    axis.set_xlabel('Época')
    axis.legend()
plt.savefig(figures_dir / 'glove_classification_losses.png', dpi=150, bbox_inches='tight')
plt.show()
reference_validation

In [ ]:
# Experimento de fracción de datos con la misma selección estratificada que usan los demás modelos
glove_choice = reference_validation[reference_validation['initialization'] == 'glove'].iloc[0]['variant']
fractions = [0.01, 0.25, 0.5, 1.0] if RUN_FULL else [1.0]
reference_fraction_rows = []
for fraction in fractions:
    indices, _ = stratified_fraction(list(range(len(train_labels))), train_labels, fraction, seed=config.seed)
    if fraction == 1.0:
        glove_row = reference_validation[reference_validation['variant'] == glove_choice].iloc[0].to_dict()
        tfidf_row = reference_validation[reference_validation['variant'] == 'tfidf_logreg'].iloc[0].to_dict()
    else:
        seed_everything(config.seed)
        model = pretrained_classifier(glove_classifier_vocabulary, glove_table, freeze=glove_choice == 'glove_frozen')
        subset_loader = make_loader([glove_encoded['train'][index] for index in indices], shuffle=True)
        history_rows = train_classifier(model, subset_loader, glove_validation_loader, epochs=classifier_epochs, device=device)
        glove_row = {**count_model_parameters(model), **evaluate_classifier(model, glove_validation_loader, device=device),
                     'training_seconds': sum(row['duration_seconds'] for row in history_rows)}
        subset_tokens = [tokenized['train'][index] for index in indices]
        subset_labels = [train_labels[index] for index in indices]
        baseline = TfidfBaseline(regularization=best_regularization, seed=config.seed).fit(subset_tokens, subset_labels)
        probabilities = baseline.predict_proba(tokenized['validation'])
        tfidf_row = {**baseline.parameter_counts(), **classification_metrics(split_labels['validation'], baseline.classifier.classes_[probabilities.argmax(axis=1)]),
                     'loss': float(log_loss(split_labels['validation'], probabilities, labels=baseline.classifier.classes_)),
                     'training_seconds': baseline.training_seconds}
    for initialization, row in (('glove', glove_row), ('tfidf', tfidf_row)):
        reference_fraction_rows.append({'fraction': fraction, 'initialization': initialization,
                                        **{key: row[key] for key in ['total', 'trainable', 'training_seconds', 'loss', 'accuracy',
                                                                     'macro_precision', 'macro_recall', 'macro_f1', 'count']}})
reference_fractions = pd.DataFrame(reference_fraction_rows)
reference_fractions.to_csv(output_dir / 'classification_fractions_reference.csv', index=False)
reference_fractions

### Evaluación en test

El conjunto de test oficial se evalúa una sola vez con la mejor variante de cada inicialización de esta parte, que es el baseline TF-IDF y la mejor variante de GloVe según la validación. El resultado se guarda y en las siguientes ejecuciones se lee de los archivos sin volver a evaluar.

In [ ]:
test_path = output_dir / 'classification_test_reference.csv'
matrices_path = output_dir / 'confusion_matrices_reference.json'
if test_path.exists() and matrices_path.exists():
    reference_test = pd.read_csv(test_path)
    reference_matrices = json.loads(matrices_path.read_text(encoding='utf-8'))
else:
    glove_test_loader = make_loader([
        ([glove_classifier_vocabulary.get(token, 0) for token in document] or [0], label)
        for document, label in zip(tokenized['test'], split_labels['test'])
    ], shuffle=False)
    glove_metrics, glove_matrix = evaluate_once(glove_models[glove_choice], glove_test_loader, device)
    tfidf_probabilities = tfidf_model.predict_proba(tokenized['test'])
    tfidf_predictions = tfidf_probabilities.argmax(axis=1)
    tfidf_metrics = {'loss': float(log_loss(split_labels['test'], tfidf_probabilities, labels=[0, 1, 2, 3])),
                     **classification_metrics(split_labels['test'], tfidf_predictions)}
    reference_test = pd.DataFrame([
        {'initialization': 'tfidf', 'variant': 'tfidf_logreg', **tfidf_metrics},
        {'initialization': 'glove', 'variant': glove_choice, **glove_metrics},
    ])
    reference_matrices = {
        'tfidf': confusion_matrix(split_labels['test'], tfidf_predictions, labels=[0, 1, 2, 3]).tolist(),
        'glove': glove_matrix.tolist(),
    }
    reference_test.to_csv(test_path, index=False)
    matrices_path.write_text(json.dumps(reference_matrices, indent=2), encoding='utf-8')
reference_test

In [ ]:
# Tabla consolidada de validación y test para todas las inicializaciones, y matrices de confusión
own_validation = pd.read_csv(output_dir / 'classification_validation.csv')
own_validation['initialization'] = own_validation['variant'].map(lambda name: 'random' if name == 'random' else 'sgns')
own_test = pd.read_csv(output_dir / 'classification_test.csv')
all_validation = pd.concat([own_validation, pd.read_csv(output_dir / 'classification_validation_reference.csv')], ignore_index=True)
test_columns = ['variant', 'loss', 'accuracy', 'macro_precision', 'macro_recall', 'macro_f1']
all_test = pd.concat([own_test, reference_test], ignore_index=True)[test_columns].rename(
    columns={name: f'test_{name}' for name in test_columns if name != 'variant'})
consolidated = all_validation.merge(all_test, on='variant', how='left')
consolidated = consolidated[['initialization', 'variant', 'total', 'trainable', 'training_seconds', 'loss', 'accuracy',
                             'macro_precision', 'macro_recall', 'macro_f1', 'test_loss', 'test_accuracy',
                             'test_macro_precision', 'test_macro_recall', 'test_macro_f1']]
consolidated = consolidated.sort_values(['initialization', 'macro_f1'], ascending=[True, False])
consolidated.to_csv(output_dir / 'classification_consolidated.csv', index=False)
display(consolidated)

all_matrices = {**json.loads((output_dir / 'confusion_matrices.json').read_text(encoding='utf-8')), **reference_matrices}
figure, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for axis, name in zip(axes, ['tfidf', 'random', 'sgns', 'glove']):
    ConfusionMatrixDisplay(np.asarray(all_matrices[name]), display_labels=class_names).plot(cmap='Blues', ax=axis, colorbar=False, xticks_rotation=30)
    axis.set_title(f'Test, {name}')
plt.savefig(figures_dir / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# F1 macro frente a la fracción de entrenamiento, una curva por inicialización
all_fractions = pd.concat([pd.read_csv(output_dir / 'classification_fractions.csv'), reference_fractions], ignore_index=True)
all_fractions.to_csv(output_dir / 'classification_fractions_all.csv', index=False)
for initialization, group in all_fractions.groupby('initialization'):
    plt.plot(group['fraction'], group['macro_f1'], marker='o', label=initialization)
plt.xscale('log')
plt.xlabel('Fracción de entrenamiento de AG News')
plt.ylabel('F1 macro de validación')
plt.legend(title='Inicialización')
plt.savefig(figures_dir / 'f1_vs_fraction.png', dpi=150, bbox_inches='tight')
plt.show()
all_fractions.pivot_table(index='fraction', columns='initialization', values='macro_f1')

## Evaluación final de similitud léxica y tabla comparativa

Hasta aquí las configuraciones se eligieron solo con WordSim-353 y las analogías de `questions-words.txt`. Con las configuraciones ya cerradas se calcula por primera vez la correlación de Spearman en SimLex-999, y se arma la tabla que reúne los tres conjuntos de vectores.

In [ ]:
simlex_pairs = load_similarity_pairs(datapath('simlex999.txt'))
similarity_rows = []
for model_name in model_spaces:
    for benchmark_name, pairs in {'wordsim353': wordsim_pairs, 'simlex999': simlex_pairs}.items():
        for scope, space in (('shared', shared_spaces[model_name]), ('own', model_spaces[model_name])):
            similarity_rows.append({'model': model_name, 'benchmark': benchmark_name, 'scope': scope,
                                    **spearman_similarity(space[0], space[1], pairs)})
similarity_final = pd.DataFrame(similarity_rows)
similarity_final.to_csv(output_dir / 'similarity_final.csv', index=False)
similarity_final.pivot_table(index=['benchmark', 'scope'], columns='model', values='spearman')

In [ ]:
# Tabla comparativa entre SGNS, gensim y GloVe
sgns_metadata = json.loads((output_dir / 'sgns_dim_100' / 'metadata.json').read_text(encoding='utf-8'))
gensim_run = json.loads((gensim_dir / 'corpus_100' / 'summary.json').read_text(encoding='utf-8'))

# Datos de entrenamiento: SGNS y gensim salen de sus metadatos. El hardware de SGNS se anota a mano porque la
# corrida no lo registró. GloVe se describe con lo que reportan sus autores (6 mil millones de tokens).
sgns_hardware = 'GPU NVIDIA (modelo no registrado en la corrida)'
# Según Pennington et al. (2014) la matriz de co-ocurrencia tomó unos 85 minutos con un hilo y, para 300 dimensiones,
# una iteración tomó 14 minutos. Los autores no dan el tiempo total de la versión de 100 dimensiones, que usa 50 iteraciones.
glove_authors = {'tokens': 6_000_000_000, 'training_seconds': float('nan'),
                 'hardware': 'Dos Xeon E5-2658 de 2.1 GHz, 32 núcleos (reportado por los autores)'}
training_info = {
    'sgns': {'tokens': sgns_metadata['tokens'], 'training_seconds': sgns_metadata['total_time_seconds'], 'hardware': sgns_hardware},
    'gensim': {'tokens': gensim_run['tokens'], 'training_seconds': gensim_run['total_time_seconds'], 'hardware': hardware_cpu},
    'glove': glove_authors,
}
accuracy_lookup = analogy_benchmark.set_index(['model', 'method', 'category'])['accuracy']
similarity_lookup = similarity_final[similarity_final['scope'] == 'shared'].set_index(['model', 'benchmark'])['spearman']
oov_lookup = oov_table.pivot_table(index='split', columns='embedding', values='oov_rate')
test_f1 = {}
for _, row in pd.concat([own_test, reference_test], ignore_index=True).iterrows():
    test_f1[row['initialization']] = row['macro_f1']

comparison_rows = []
for model_name, (vectors, vocab) in model_spaces.items():
    oov_column = 'glove' if model_name == 'glove' else 'sgns_and_gensim'
    row = {'model': model_name, **training_info[model_name], 'vocabulary_size': len(vocab), 'dimension': vectors.shape[1]}
    for method in ('3cosadd', '3cosmul'):
        for category in ('semantic', 'syntactic', 'total'):
            row[f'{method}_{category}'] = accuracy_lookup[(model_name, method, category)]
    row['wordsim353_spearman'] = similarity_lookup[(model_name, 'wordsim353')]
    row['simlex999_spearman'] = similarity_lookup[(model_name, 'simlex999')]
    row['parallelism_mean_cosine'] = parallelism_summary.loc[model_name, 'mean_cosine']
    row['ag_news_oov_train'] = oov_lookup.loc['train', oov_column]
    row['ag_news_oov_test'] = oov_lookup.loc['test', oov_column]
    row['ag_news_test_macro_f1'] = test_f1.get(model_name, float('nan'))
    comparison_rows.append(row)
model_comparison = pd.DataFrame(comparison_rows).set_index('model')
model_comparison.to_csv(output_dir / 'model_comparison.csv')
model_comparison.T

In [ ]:
# Accuracy de analogías frente al número de tokens del corpus, con GloVe como referencia.
# Todas las cifras usan el vocabulario propio de cada modelo. GloVe se restringe a sus palabras más frecuentes
# para tener un número de candidatos parecido al de SGNS.
glove_reference_path = output_dir / 'glove_reference_metrics.json'
if glove_reference_path.exists():
    glove_reference = json.loads(glove_reference_path.read_text(encoding='utf-8'))
else:
    glove_matrix_small, glove_vocabulary_small = restrict_space(glove_vectors, glove_vocabulary, glove.index_to_key[:len(vocabulary)])
    glove_reference = epoch_evaluation(glove_matrix_small, glove_vocabulary_small, wordsim_pairs, analogy_records)
    glove_reference.pop('neighbors')
    glove_reference_path.write_text(json.dumps(glove_reference, indent=2), encoding='utf-8')

sgns_summary = pd.read_csv(output_dir / 'sgns_summary.csv')
sgns_point = sgns_summary[sgns_summary['dimension'] == final_dimension].iloc[0]
corpus_runs = gensim_summary[gensim_summary['run'].str.startswith('corpus_')].sort_values('tokens')
plt.plot(corpus_runs['tokens'], corpus_runs['total_accuracy'], marker='o', label='gensim')
plt.scatter([sgns_metadata['tokens']], [sgns_point['total_accuracy']], color='tab:red', zorder=3, label='SGNS')
plt.axhline(glove_reference['total_accuracy'], color='gray', linestyle='--', label='GloVe (6 mil millones de tokens)')
plt.xscale('log')
plt.xlabel('Tokens del corpus de entrenamiento')
plt.ylabel('Accuracy total de analogías (3CosAdd)')
plt.legend()
plt.savefig(figures_dir / 'accuracy_vs_tokens.png', dpi=150, bbox_inches='tight')
plt.show()
pd.concat([
    corpus_runs[['run', 'tokens', 'total_accuracy']],
    pd.DataFrame([{'run': 'sgns', 'tokens': sgns_metadata['tokens'], 'total_accuracy': sgns_point['total_accuracy']},
                  {'run': 'glove', 'tokens': glove_authors['tokens'], 'total_accuracy': glove_reference['total_accuracy']}]),
], ignore_index=True)